In [3]:
import pandas as pd
import sys

sys.path.append('./..')

In [4]:
from tqdm import tqdm
import os
import time

import pandas as pd
import numpy as np

from src.loader_functions.get_liquipedia_tournaments import get_liquipedia_tournaments
from src.loader_functions.get_liquipedia_tornament_info import get_liquipedia_tornament_info
from src.loader_functions.get_aligulac_matches import get_aligulac_matches
from src.loader_functions.json_functions import load_processed_tournaments, save_processed_tournaments

from src.config import LIQUIPEDIA_URLS, SAVE_PATH, SAVE_PATH_TOURNAMENTS

In [6]:
df = pd.read_csv('./../data/dataset.csv', sep=';')

tournament_counts = (
    df.groupby('tournament')
      .size()
      .to_dict()
)


In [8]:
def build_processed_tournaments_from_csv(df: pd.DataFrame) -> dict:
    return {
        tournament: {
            "matches": int(count)
        }
        for tournament, count in (
            df.groupby("tournament").size().items()
        )
    }
# end def

processed_tournaments = build_processed_tournaments_from_csv(df)
save_processed_tournaments('./../data/tournaments_update.json', processed_tournaments)

In [9]:
existing_tournaments = set(
    df.loc[df['tournament'].notna(), 'tournament'].unique()
)

In [ ]:
for url, tier in LIQUIPEDIA_URLS.items():
    print(f'start : {url}')
    tournaments = get_liquipedia_tournaments(url)

    time.sleep(30 + np.random.uniform(10, 30))

    for tournament in tqdm(tournaments):

        if tournament in existing_tournaments:
            continue

        aligulac_url = get_liquipedia_tornament_info(tournament)
        time.sleep(30 + np.random.uniform(10, 30))

        if aligulac_url is None:
            continue

        if aligulac_url in df['url'].unique():
            continue

        aligulac_data = get_aligulac_matches(aligulac_url)

        if aligulac_data.empty:
            continue

        aligulac_data['tier'] = tier
        aligulac_data['url'] = aligulac_url
        aligulac_data['tournament'] = tournament

        df = pd.concat([df, aligulac_data], ignore_index=True)
        df.to_csv(SAVE_PATH, sep=';', index=False)

        existing_tournaments.add(tournament)


start : https://liquipedia.net/starcraft2/S-Tier_Tournaments


  1%|          | 1/96 [00:57<1:31:49, 57.99s/it]